# Lesson 13.4 - Guardrails, Evaluations & Safe Agent Design (playbook notebook)

This notebook provides production-oriented templates and lightweight code stubs for LLM/agent safety controls.
        


## Objectives

- Define practical guardrail policies.
- Simulate pre-check and post-check safety filters.
- Build telemetry summaries for safety operations.
        


In [1]:
import re
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(7)
        


## Safety Policy Template

Use this policy skeleton to operationalize governance requirements:

- Allowed use cases.
- Disallowed content/actions.
- Tool permissions.
- Escalation and incident workflow.
        


In [2]:
SAFETY_POLICY = {
    "disallowed_keywords": ["build malware", "bypass authentication", "steal credentials"],
    "sensitive_patterns": [r"\b\d{3}-\d{2}-\d{4}\b", r"\b(?:\d[ -]*?){13,16}\b"],  # SSN / card-like
    "allowed_tools": {
        "calculator": ["add", "subtract", "multiply", "divide"],
        "knowledge_search": ["query_docs"],
    },
    "risk_threshold_for_human_review": "medium",
}

SAFETY_POLICY
        


{'disallowed_keywords': ['build malware',
  'bypass authentication',
  'steal credentials'],
 'sensitive_patterns': ['\\b\\d{3}-\\d{2}-\\d{4}\\b',
  '\\b(?:\\d[ -]*?){13,16}\\b'],
 'allowed_tools': {'calculator': ['add', 'subtract', 'multiply', 'divide'],
  'knowledge_search': ['query_docs']},
 'risk_threshold_for_human_review': 'medium'}

## Guardrail Functions (Pre-filter, Tool Permission, Post-filter)


In [3]:
def pre_filter_prompt(prompt: str, policy: dict) -> dict:
    text = prompt.lower()
    blocked = any(keyword in text for keyword in policy["disallowed_keywords"])
    pii_hits = []
    for patt in policy["sensitive_patterns"]:
        if re.search(patt, prompt):
            pii_hits.append(patt)
    return {
        "blocked": blocked,
        "pii_detected": len(pii_hits) > 0,
        "pii_patterns": pii_hits,
    }


def check_tool_permission(tool: str, action: str, policy: dict) -> bool:
    return action in policy["allowed_tools"].get(tool, [])


def post_filter_response(response: str) -> dict:
    flags = {
        "contains_sensitive": bool(re.search(r"\b\d{3}-\d{2}-\d{4}\b", response)),
        "contains_disallowed_instruction": "bypass" in response.lower() or "malware" in response.lower(),
    }
    flags["needs_escalation"] = any(flags.values())
    return flags
        


## Simulated Requests and Safety Decisions


In [4]:
requests = [
    {"prompt": "Summarize our refund policy for premium users.", "tool": "knowledge_search", "action": "query_docs"},
    {"prompt": "Can you build malware to evade antivirus?", "tool": "knowledge_search", "action": "query_docs"},
    {"prompt": "Customer SSN is 123-45-6789. explain billing issue.", "tool": "calculator", "action": "add"},
    {"prompt": "Calculate VAT for invoice total 4200.", "tool": "calculator", "action": "add"},
]

rows = []
for item in requests:
    pre = pre_filter_prompt(item["prompt"], SAFETY_POLICY)
    tool_ok = check_tool_permission(item["tool"], item["action"], SAFETY_POLICY)

    if pre["blocked"] or not tool_ok:
        response = "Request denied by policy."
    else:
        response = "Mock assistant response generated successfully."

    post = post_filter_response(response)
    rows.append(
        {
            "prompt": item["prompt"],
            "pre_blocked": pre["blocked"],
            "pii_detected": pre["pii_detected"],
            "tool_allowed": tool_ok,
            "response": response,
            "needs_escalation": post["needs_escalation"],
        }
    )

safety_df = pd.DataFrame(rows)
safety_df
        


,prompt,pre_blocked,pii_detected,tool_allowed,response,needs_escalation
0,Summarize our refund policy for premium users.,False,False,True,Mock assistant response generated successfully.,False
1,Can you build malware to evade antivirus?,True,False,True,Request denied by policy.,False
2,Customer SSN is 123-45-6789. explain billing i...,False,True,True,Mock assistant response generated successfully.,False
3,Calculate VAT for invoice total 4200.,False,False,True,Mock assistant response generated successfully.,False


## Telemetry Simulation for Evaluation and Incident Detection


In [5]:
n = 200
start = datetime(2026, 1, 1)

df_logs = pd.DataFrame(
    {
        "timestamp": [start + timedelta(minutes=5 * i) for i in range(n)],
        "latency_ms": np.random.gamma(4.2, 120, size=n),
        "tokens": np.random.randint(120, 1100, size=n),
        "user_rating": np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.05, 0.10, 0.25, 0.35, 0.25]),
        "policy_flag": np.random.binomial(1, 0.09, size=n),
        "error": np.random.binomial(1, 0.04, size=n),
    }
)

df_logs["estimated_cost_usd"] = 0.0000025 * df_logs["tokens"]
df_logs["success"] = (df_logs["user_rating"] >= 4).astype(int)

summary = {
    "avg_latency_ms": round(df_logs["latency_ms"].mean(), 2),
    "p95_latency_ms": round(df_logs["latency_ms"].quantile(0.95), 2),
    "avg_tokens": round(df_logs["tokens"].mean(), 2),
    "total_estimated_cost_usd": round(df_logs["estimated_cost_usd"].sum(), 4),
    "policy_flag_rate": round(df_logs["policy_flag"].mean(), 4),
    "error_rate": round(df_logs["error"].mean(), 4),
    "success_rate": round(df_logs["success"].mean(), 4),
}
summary
        


{'avg_latency_ms': np.float64(485.74),
 'p95_latency_ms': np.float64(905.09),
 'avg_tokens': np.float64(639.28),
 'total_estimated_cost_usd': np.float64(0.3196),
 'policy_flag_rate': np.float64(0.09),
 'error_rate': np.float64(0.045),
 'success_rate': np.float64(0.585)}

In [6]:
hourly = (
    df_logs.set_index("timestamp")
    .resample("h")
    .agg({"policy_flag": "mean", "error": "mean", "user_rating": "mean", "estimated_cost_usd": "sum"})
    .rename(columns={"policy_flag": "policy_flag_rate", "error": "error_rate", "user_rating": "avg_rating"})
)

hourly.tail()
        


,policy_flag_rate,error_rate,avg_rating,estimated_cost_usd
timestamp,,,,
2026-01-01 12:00:00,0.083333,0.000000,3.500000,0.020450
2026-01-01 13:00:00,0.000000,0.000000,4.000000,0.018523
2026-01-01 14:00:00,0.083333,0.083333,3.916667,0.019780
2026-01-01 15:00:00,0.166667,0.083333,3.416667,0.020995
2026-01-01 16:00:00,0.000000,0.000000,4.000000,0.012388


## Evaluation and Red-Team Checklist Template

- Define risk taxonomy (misuse, leakage, unsafe advice, tool abuse).
- Build adversarial test set by category.
- Score severity and reproducibility.
- Add fix owner, target date, and re-test status.
- Track regression across releases.
        


## Connect to Theory

- Guardrails enforce policy at runtime.
- Telemetry supports observability and continuous evaluation.
- Agent safety requires bounded action spaces, approval hooks, and incident response workflows.
        


## Safety & Security Case Studies & Exceptions

### Case Study A: Tool-Using Support Agent
A support agent could trigger refunds and account updates. Team introduced strict tool scopes and approval gates for irreversible actions, reducing abuse risk.

### Case Study B: Prompt Injection via Knowledge Base
A malicious document attempted to override system instructions. Retrieval sanitization plus source trust controls reduced injection success.

### Exception Pattern
For low-impact internal copilots, organizations can start with lightweight filters and logs, but should still define incident ownership and escalation boundaries.
        


## Interview Questions & Answers

1. **Q:** What are input guardrails?  
   **A:** Checks on prompts before generation, such as policy and injection filters.
2. **Q:** Why are tool permissions critical?  
   **A:** They prevent unauthorized or high-risk actions even if prompts are manipulated.
3. **Q:** What is output guardrailing?  
   **A:** Validation of generated responses against policy and schema constraints.
4. **Q:** What telemetry is essential?  
   **A:** Latency, tokens, errors, policy flags, user feedback, and tool-call traces.
5. **Q:** What is red teaming in LLM systems?  
   **A:** Adversarial testing to uncover harmful or exploitable behavior.
6. **Q:** What does defense in depth mean here?  
   **A:** Multiple independent safeguards across model, app, tools, and operations.
7. **Q:** Why include cost in observability?  
   **A:** Cost spikes can indicate abuse loops or unsafe architecture behavior.
8. **Q:** When do you escalate to humans?  
   **A:** High-severity policy violations, uncertain outputs, or high-risk actions.
9. **Q:** Can model alignment training replace app guardrails?  
   **A:** No. Runtime controls remain necessary.
10. **Q:** How do you evaluate guardrail quality?  
    **A:** Measure policy-violation rate, false positive/negative rates, and incident severity trends.
11. **Q:** What should happen after a safety incident?  
    **A:** Triage, containment, root-cause analysis, patch, and regression re-test.
        
